In [ ]:
# ===============================
# IMPORT LIBRARIES
# ===============================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ===============================
# STEP 1: LOAD GENERATOR MODEL
# ===============================

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Model ready!")


# ===============================
# STEP 2: GENERATE ANSWER FUNCTION
# ===============================

def generate_answer(question):

    prompt = f"""
You are an AI assistant explaining machine learning concepts.

Answer clearly in 3–5 sentences.
Explain in simple words.

Question: {question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer.strip()


# ===============================
# STEP 3: INTERACTIVE MODE
# ===============================

print("\nNon-RAG system ready! Ask questions.\n")

while True:
    question = input("Ask: ")

    if question.lower() == "exit":
        break

    answer = generate_answer(question)

    print("\nAnswer:\n", answer)

Loading model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model ready!

Non-RAG system ready! Ask questions.


Answer:
 The number of digits in a digit is 0

Answer:
 a machine learning tutorial


In [ ]:
# ===============================
# IMPORT LIBRARIES
# ===============================

!pip install pypdf
!pip install faiss-cpu
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ===============================
# STEP 1: LOAD PDF
# ===============================

file_path = "2019BurkovTheHundred-pageMachineLearning.pdf"

def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

print("Loading book...")
book_text = load_pdf(file_path)
print("Book loaded!")
print("Characters:", len(book_text))


# ===============================
# STEP 2: SPLIT TEXT INTO CHUNKS
# ===============================

# ✅ smaller chunks improve accuracy
def chunk_text(text, chunk_size=220):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_text(book_text)
print("Chunks created:", len(chunks))


# ===============================
# STEP 3: CREATE EMBEDDINGS
# ===============================

print("Creating embeddings...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunks, show_progress_bar=True)


# ===============================
# STEP 4: STORE IN FAISS
# ===============================

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print("FAISS index ready!")


# ===============================
# STEP 5: LOAD GENERATOR MODEL
# ===============================

print("Loading generator model...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Generator ready!")

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ===============================
# STEP 6: QUESTION FUNCTION
# ===============================

def ask_question(question):

    q_embed = embed_model.encode([question])

    # ✅ retrieve fewer chunks to avoid mixing topics
    distances, indices = index.search(np.array(q_embed), k=3)

    context = " ".join([chunks[i] for i in indices[0]])[:1200]

    prompt = f"""
You are an AI assistant answering questions from a machine learning textbook.

Use the context to give a COMPLETE and CLEAR explanation.

Answer in 3–5 sentences.
Explain in simple words.

Context:
{context}

Question: {question}

Answer:
"""

    answer = generate_answer(prompt)

    return answer.strip()


# ===============================
# STEP 7: INTERACTIVE MODE
# ===============================

print("\nRAG system ready! Ask questions.\n")

while True:
    question = input("Ask: ")

    if question.lower() == "exit":
        break

    answer = ask_question(question)

    print("\nAnswer:\n", answer)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.7 MB/s eta 0:00:00
Loading book...
Book loaded!
Characters: 279922
Chunks created: 224
Creating embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready!

RAG system ready! Ask questions.


Answer:
 Overspent paper

Answer:
 The data for supervised learning is a collection of pairs (input, output).
